# ☕🗺️ Mapify: Google Maps Cafe Photo Harvester (Kaggle Edition)

سامانه تمام‌خودکار و امن جهت استخراج آدرس تصاویر با کیفیت اورجینال (`s0-rw`) و شناسه‌های یکتای گوگل مپ (`place_id`, `cid`, `fid`, `kgmid`) برای کافه‌های ایران.

### 🚀 راهنمای سریع اجرا در کگل (Quickstart):
1. در پنل تنظیمات سمت راست نوت‌بوک کگل (Notebook Options)، گزینه **Internet** را روی **Internet On** قرار دهید.
2. دیتابیس کافه‌ها (`1405-08-12-cafochino.db` یا هر فایل `.db` دیگر) را از بخش **+ Add Input** اضافه کنید.
3. برای اجرای پس‌زمینه (Background Run)، دکمه **Save Version** (گزینه `Save & Run All (Commit)`) را بزنید.
4. خروجی در `/kaggle/working/cafe_photos.db` ذخیره شده و پس از اتمام قابل دانلود است.

In [ ]:
# 1. نصب پکیج‌های ضروری و مرورگر کرومیوم با تمامی وابستگی‌های لینوکس
!pip install -q playwright playwright-stealth loguru tqdm
!playwright install chromium
!playwright install-deps chromium
print('✅ Playwright Chromium and dependencies installed successfully!')

In [ ]:
# 2. تنظیمات و پیدا کردن خودکار دیتابیس ورودی در کگل
import os
import glob
from dataclasses import dataclass, field
from pathlib import Path

# جستجوی خودکار دیتابیس در فولدرهای اینپوت کگل
possible_dbs = glob.glob('/kaggle/input/**/*.db', recursive=True)
if possible_dbs:
    DEFAULT_SOURCE_DB = possible_dbs[0]
    print(f'🔍 Auto-detected source database: {DEFAULT_SOURCE_DB}')
else:
    # مسیر پیش‌فرض در صورت عدم وجود اینپوت
    DEFAULT_SOURCE_DB = 'cafochino.db'
    print(f'⚠️ No .db found in /kaggle/input, fallback to local: {DEFAULT_SOURCE_DB}')

@dataclass
class KaggleConfig:
    # مسیرهای فایل
    SOURCE_DB_PATH: str = DEFAULT_SOURCE_DB
    OUTPUT_DB_PATH: str = '/kaggle/working/cafe_photos.db'
    
    # تنظیمات شاردینگ برای اجرای موازی در چند نوت‌بوک (مثلا 4 نوت‌بوک موازی)
    SHARD_ID: int = None       # شماره شارد (1 تا TOTAL_SHARDS). در صورت None کل کافه‌ها پردازش می‌شوند
    TOTAL_SHARDS: int = None   # تعداد کل شاردها (مثلا 4)
    LIMIT_CAFES: int = None    # محدودیت تعداد (برای تست مثلا 50، برای اجرا روی همه None بگذارید)
    
    # شعاع جستجو و دقت مچینگ
    ZOOM_LEVEL: int = 17
    MATCH_RADIUS_METERS: float = 300.0
    MATCH_THRESHOLD: float = 0.75
    
    # مکث‌های تصادفی انسانی برای جلوگیری ۱۰۰٪ از ریت‌لیمیت
    MIN_DELAY_PER_CAFE: float = 4.5
    MAX_DELAY_PER_CAFE: float = 8.5
    CAFES_PER_BATCH: int = 35
    BATCH_COOLDOWN_SECONDS: float = 120.0
    RECYCLE_CONTEXT_EVERY: int = 20
    
    # کیفیت و سایز عکس (s0-rw = اورجینال دوربین با فرمت WebP)
    PHOTO_MODIFIER: str = '=s0-rw'
    
    # کوکی‌های رضایت حریم خصوصی جهت رد کردن دیواره‌های GDPR
    CONSENT_COOKIES: list = field(default_factory=lambda: [
        {'name': 'SOCS', 'value': 'CAESNQgDEitwSgJBUhADGgEGATAW', 'domain': '.google.com', 'path': '/'},
        {'name': 'CONSENT', 'value': 'YES+', 'domain': '.google.com', 'path': '/'}
    ])

config = KaggleConfig()
print('✅ Configuration initialized!')

In [ ]:
# 3. لایه دیتابیس و استخراج شناسه‌های یکتای مکان (place_id, cid, fid, kgmid)
import base64
import re
import sqlite3
import struct
import time
from typing import Optional, List, Dict, Any
from urllib.parse import unquote

@dataclass
class CafeRecord:
    id: int
    name: str
    latitude: float
    longitude: float
    city: Optional[str] = None
    address: Optional[str] = None
    phone: Optional[str] = None
    instagram: Optional[str] = None
    website: Optional[str] = None

def extract_place_identifiers(url: str) -> Dict[str, Optional[str]]:
    """تبدیل قطعی و بدون خطای شناسه‌های URL به Google Place ID (ChIJ...), Decimal CID, FID و KGMID"""
    ids = {'place_id': None, 'cid': None, 'fid': None, 'kgmid': None}
    if not url:
        return ids

    m_hex = re.search(r'1s(0x[0-9a-fA-F]+):(0x[0-9a-fA-F]+)', url)
    if m_hex:
        hex1, hex2 = m_hex.group(1), m_hex.group(2)
        ids['fid'] = f'{hex1}:{hex2}'
        try:
            val1 = int(hex1, 16)
            val2 = int(hex2, 16)
            ids['cid'] = str(val2)
            # ساختار باینری پروتوباف جهت بازتولید Place ID استاندارد گوگل
            payload = b'\n\x12\t' + struct.pack('<Q', val1) + b'\x11' + struct.pack('<Q', val2)
            ids['place_id'] = base64.urlsafe_b64encode(payload).decode('ascii').rstrip('=')
        except Exception:
            pass

    m_pid = re.search(r'[?&](?:query_place_id|place_id)=([a-zA-Z0-9_\-]+)', url)
    if m_pid and not ids['place_id']:
        ids['place_id'] = m_pid.group(1)

    m_cid = re.search(r'[?&]cid=(\d+)', url)
    if m_cid and not ids['cid']:
        ids['cid'] = m_cid.group(1)

    m_kgmid = re.search(r'16s(?:%2F|/)([^?&!]+)', url)
    if m_kgmid:
        val = unquote(m_kgmid.group(1))
        ids['kgmid'] = val if val.startswith('/') else f'/{val}'

    return ids

class DatabaseManager:
    def __init__(self, source_db_path: str, output_db_path: str):
        self.source_db_path = source_db_path
        self.output_db_path = output_db_path
        self._init_output_db()

    def _get_connection(self, db_path: str, read_only: bool = False) -> sqlite3.Connection:
        if read_only:
            abs_path = os.path.abspath(db_path).replace('\\', '/')
            uri = f'file:{abs_path}?mode=ro'
            conn = sqlite3.connect(uri, uri=True, timeout=60.0)
        else:
            os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
            conn = sqlite3.connect(db_path, timeout=60.0)
            conn.execute('PRAGMA journal_mode = WAL;')
            conn.execute('PRAGMA synchronous = NORMAL;')
        conn.row_factory = sqlite3.Row
        return conn

    def _init_output_db(self) -> None:
        conn = self._get_connection(self.output_db_path)
        cur = conn.cursor()
        cur.execute('''
        CREATE TABLE IF NOT EXISTS places (
            cafe_id INTEGER PRIMARY KEY,
            place_id TEXT,
            cid TEXT,
            fid TEXT,
            kgmid TEXT,
            title TEXT,
            searched_name TEXT,
            searched_lat REAL,
            searched_lng REAL,
            scraped_lat REAL,
            scraped_lng REAL,
            address TEXT,
            status TEXT,
            name_score REAL,
            distance_meters REAL,
            photos_count INTEGER DEFAULT 0,
            search_url TEXT,
            google_maps_url TEXT,
            scraped_at TEXT
        );
        ''')
        cur.execute('''
        CREATE TABLE IF NOT EXISTS place_photos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            cafe_id INTEGER NOT NULL,
            photo_url TEXT NOT NULL,
            high_res_url TEXT NOT NULL,
            source_type TEXT DEFAULT 'gallery',
            author_name TEXT,
            created_at TEXT,
            UNIQUE(cafe_id, photo_url),
            FOREIGN KEY (cafe_id) REFERENCES places(cafe_id)
        );
        ''')
        cur.execute('CREATE INDEX IF NOT EXISTS idx_places_status ON places(status);')
        cur.execute('CREATE INDEX IF NOT EXISTS idx_places_place_id ON places(place_id);')
        cur.execute('CREATE INDEX IF NOT EXISTS idx_photos_cafe ON place_photos(cafe_id);')
        conn.commit()
        conn.close()

    def get_already_processed_ids(self) -> set:
        conn = self._get_connection(self.output_db_path, read_only=True)
        cur = conn.cursor()
        rows = cur.execute('SELECT cafe_id FROM places').fetchall()
        processed = {r['cafe_id'] for r in rows}
        conn.close()
        return processed

    def load_pending_cafes(self, limit: Optional[int] = None, shard_id: Optional[int] = None, total_shards: Optional[int] = None) -> List[CafeRecord]:
        processed_ids = self.get_already_processed_ids()
        conn = self._get_connection(self.source_db_path, read_only=True)
        cur = conn.cursor()
        query = '''
        SELECT id, name, latitude, longitude, city, address, phone, instagram, website
        FROM cafes_cafe
        WHERE latitude IS NOT NULL AND longitude IS NOT NULL
        ORDER BY id ASC
        '''
        rows = cur.execute(query).fetchall()
        conn.close()

        pending = []
        for r in rows:
            cid = r['id']
            if cid in processed_ids:
                continue
            if total_shards and shard_id:
                if (cid % total_shards) != (shard_id - 1):
                    continue
            pending.append(CafeRecord(
                id=cid, name=r['name'], latitude=float(r['latitude']), longitude=float(r['longitude']),
                city=r['city'], address=r['address'], phone=r['phone'], instagram=r['instagram'], website=r['website']
            ))
            if limit and len(pending) >= limit:
                break
        return pending

    def save_place_and_photos(self, cafe: CafeRecord, status: str, title: Optional[str] = None, scraped_lat: Optional[float] = None, scraped_lng: Optional[float] = None, name_score: float = 0.0, distance_meters: Optional[float] = None, search_url: str = '', google_maps_url: Optional[str] = None, photos: Optional[List[Dict[str, Any]]] = None) -> None:
        photos = photos or []
        extracted = extract_place_identifiers(google_maps_url or '')
        conn = self._get_connection(self.output_db_path)
        cur = conn.cursor()
        now_iso = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

        cur.execute('''
        INSERT OR REPLACE INTO places (
            cafe_id, place_id, cid, fid, kgmid, title, searched_name, searched_lat, searched_lng,
            scraped_lat, scraped_lng, address, status, name_score, distance_meters,
            photos_count, search_url, google_maps_url, scraped_at
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            cafe.id, extracted['place_id'], extracted['cid'], extracted['fid'], extracted['kgmid'],
            title or cafe.name, cafe.name, cafe.latitude, cafe.longitude,
            scraped_lat, scraped_lng, cafe.address, status, round(name_score, 2),
            round(distance_meters, 1) if distance_meters is not None else None,
            len(photos), search_url, google_maps_url, now_iso
        ))

        for p in photos:
            cur.execute('''
            INSERT OR IGNORE INTO place_photos (
                cafe_id, photo_url, high_res_url, source_type, author_name, created_at
            ) VALUES (?, ?, ?, ?, ?, ?)
            ''', (cafe.id, p['photo_url'], p.get('high_res_url', p['photo_url']), p.get('source_type', 'gallery'), p.get('author_name'), now_iso))

        conn.commit()
        conn.close()

print('✅ Database Manager and Place Identifiers ready!')

In [ ]:
# 4. موتور تطبیق پیشرفته نام‌های فارسی، کلمات مرکب و فاصله جغرافیایی ۳۰۰ متری
import math
from difflib import SequenceMatcher

STOP_WORDS = {
    'کافه', 'رستوران', 'کافی', 'شاپ', 'قهوه', 'بار', 'لانژ', 'فست', 'فود', 'عمارت', 'خانه', 'سرای',
    'cafe', 'café', 'restaurant', 'coffee', 'shop', 'bar', 'lounge', 'fast', 'food', 'bakery',
    'شعبه', 'ایرانی', 'سنتی', 'مدرن', 'اصلی', 'مرکزی', 'بین', 'الملل', 'بین‌المللی',
    'تهران', 'اصفهان', 'مشهد', 'شیراز', 'تبریز', 'اهواز', 'کرج', 'قم', 'رشت', 'کیش'
}

PERSIAN_TO_LATIN = {
    'آ': 'a', 'ا': 'a', 'ب': 'b', 'پ': 'p', 'ت': 't', 'ث': 's', 'ج': 'j', 'چ': 'ch', 'ح': 'h', 'خ': 'kh',
    'د': 'd', 'ذ': 'z', 'ر': 'r', 'ز': 'z', 'ژ': 'zh', 'س': 's', 'ش': 'sh', 'ص': 's', 'ض': 'z', 'ط': 't',
    'ظ': 'z', 'ع': 'a', 'غ': 'gh', 'ف': 'f', 'ق': 'gh', 'ک': 'k', 'گ': 'g', 'ل': 'l', 'م': 'm', 'ن': 'n',
    'و': 'o', 'ه': 'h', 'ی': 'i', 'ي': 'i', 'ك': 'k', 'ة': 'h', 'ء': '', 'ئ': 'i', 'ؤ': 'o'
}

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    if None in (lat1, lon1, lat2, lon2):
        return float('inf')
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2.0)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2.0)**2
    return R * (2.0 * math.atan2(math.sqrt(a), math.sqrt(max(0.0, 1.0 - a))))

def normalize_text(text: str) -> str:
    if not text:
        return ''
    t = text.lower().replace('ي', 'ی').replace('ك', 'ک').replace('ة', 'ه').replace('آ', 'ا')
    t = re.sub(r'[\u064B-\u065F\u0670]', '', t)
    for k, v in {'۰':'0','۱':'1','۲':'2','۳':'3','۴':'4','۵':'5','۶':'6','۷':'7','۸':'8','۹':'9'}.items():
        t = t.replace(k, v)
    t = re.sub(r'[\(\)\[\]\{\}\-_,\.\+:;/\&\\\|\'\"`!?]', ' ', t)
    return ' '.join(t.split())

def clean_tokens(text: str) -> List[str]:
    tokens = normalize_text(text).split()
    filtered = [w for w in tokens if w not in STOP_WORDS]
    return filtered if filtered else tokens

def compute_name_similarity(name1: str, name2: str) -> float:
    tok1, tok2 = clean_tokens(name1), clean_tokens(name2)
    direct_ratio = SequenceMatcher(None, ' '.join(tok1), ' '.join(tok2)).ratio()
    joined_ratio = SequenceMatcher(None, ''.join(tok1), ''.join(tok2)).ratio()  # کلمات سرهم مثل پروچلو vs پرو چلو
    
    lat1 = ''.join(PERSIAN_TO_LATIN.get(c, c) for c in ''.join(tok1))
    lat2 = ''.join(PERSIAN_TO_LATIN.get(c, c) for c in ''.join(tok2))
    phonetic_ratio = SequenceMatcher(None, lat1, lat2).ratio() if (lat1 and lat2) else 0.0
    return round(max(direct_ratio, joined_ratio, phonetic_ratio), 2)

def evaluate_candidate(target_name, target_lat, target_lng, cand_title, cand_lat, cand_lng, max_radius=300.0, match_threshold=0.75):
    name_score = compute_name_similarity(target_name, cand_title)
    dist = haversine_distance(target_lat, target_lng, cand_lat, cand_lng) if (cand_lat and cand_lng) else float('inf')
    if dist > max_radius:
        return False, 0.0, dist, name_score, f'Exceeded {max_radius}m radius ({dist:.1f}m)'
    dist_score = 1.0 if dist <= 30.0 else max(0.0, 1.0 - 0.5 * ((dist - 30.0) / (max_radius - 30.0)))
    composite_score = round((0.60 * name_score) + (0.40 * dist_score), 2)
    is_match = (composite_score >= match_threshold) or (name_score >= 0.85 and dist <= max_radius)
    return is_match, composite_score, dist, name_score, None

print('✅ Matching Engine initialized!')

In [ ]:
# 5. موتور مرورگر Playwright Stealth و ره‌گیری ترافیک شبکه (Network Interceptor)
import asyncio
import random
from urllib.parse import quote_plus
from playwright.async_api import async_playwright
from playwright_stealth import Stealth

PHOTO_PATTERN = re.compile(r'https?://lh[0-9]\.googleusercontent\.com/(?:p|grass-cs|gpms-cs-s|gps-cs-s|gps-cs-p)/[a-zA-Z0-9_\-]+')

class PhotoInterceptor:
    def __init__(self, modifier='=s0-rw'):
        self.modifier = modifier
        self.captured = {}

    def clear(self):
        self.captured.clear()

    def add_url(self, raw_url, source='gallery'):
        if not raw_url:
            return
        dec = unquote(raw_url)
        for m in PHOTO_PATTERN.finditer(dec):
            base = m.group(0)
            if '/a/' in base or '/a-/' in base or 'default-user' in base:
                continue
            if base not in self.captured:
                self.captured[base] = {
                    'photo_url': base,
                    'high_res_url': f'{base.split("=")[0]}{self.modifier}',
                    'source_type': source
                }

    async def on_response(self, resp):
        url = resp.url
        if 'googleusercontent.com' in url:
            self.add_url(url, 'cdn')
        if any(k in url for k in ['/maps/rpc/', '/preview/place', 'batchexecute', 'photometa']):
            try:
                ct = resp.headers.get('content-type', '')
                if ('json' in ct or 'text' in ct or 'javascript' in ct) and 'image' not in ct:
                    body = await resp.text()
                    self.add_url(body, 'rpc')
            except Exception:
                pass

    def get_photos(self):
        return list(self.captured.values())

class MapifyScraper:
    def __init__(self, cfg: KaggleConfig, db: DatabaseManager):
        self.cfg = cfg
        self.db = db
        self.interceptor = PhotoInterceptor(cfg.PHOTO_MODIFIER)
        self.playwright = None
        self.browser = None
        self.context = None
        self.page = None
        self.requests_count = 0

    async def start(self):
        self.playwright = await async_playwright().start()
        self.browser = await self.playwright.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-setuid-sandbox', '--disable-dev-shm-usage', '--disable-features=IsolateOrigins,site-per-process']
        )
        await self._new_context()

    async def _new_context(self):
        if self.context:
            try: await self.context.close()
            except Exception: pass
        self.context = await self.browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36',
            locale='fa-IR',
            viewport={'width': 1280, 'height': 800}
        )
        await self.context.add_cookies(self.cfg.CONSENT_COOKIES)
        self.page = await self.context.new_page()
        await Stealth().apply_stealth_async(self.page)
        await self.page.route('**/*', lambda r: r.abort() if r.request.resource_type in ('media', 'font') else r.continue_())
        self.page.on('response', self.interceptor.on_response)
        self.page.on('request', lambda req: self.interceptor.add_url(req.url, 'request'))
        self.requests_count = 0

    async def close(self):
        if self.context: await self.context.close()
        if self.browser: await self.browser.close()
        if self.playwright: await self.playwright.stop()

    async def _wait_signal(self, max_wait=7.5):
        t0 = asyncio.get_event_loop().time()
        while (asyncio.get_event_loop().time() - t0) < max_wait:
            cur = self.page.url
            if 'google.com/sorry' in cur: return 'captcha'
            if '/maps/place/' in cur:
                h1 = self.page.locator('h1.DUwDvf, h1').first
                if await h1.is_visible(): return 'direct_place'
            count = await self.page.locator("a[href*='/maps/place/']").count()
            if count > 0:
                if '/maps/place/' in cur: return 'direct_place'
                return 'search_feed'
            if await self.page.locator("text='نتیجه‌ای یافت نشد', text='Google Maps can't find'").count() > 0:
                return 'not_found'
            await asyncio.sleep(0.25)
        return 'direct_place' if '/maps/place/' in self.page.url else ('search_feed' if await self.page.locator("a[href*='/maps/place/']").count() > 0 else 'not_found')

    async def _harvest_photos(self):
        try:
            btn = self.page.locator("button[role='tab'][aria-label*='عکس'], button[role='tab'][aria-label*='Photo'], button.AoCdqe, div[jsaction*='heroHeaderImage']").first
            if await btn.is_visible(timeout=1500):
                await btn.click()
                await self.page.wait_for_timeout(1500)
                for _ in range(3):
                    await self.page.keyboard.press('PageDown')
                    await self.page.wait_for_timeout(500)
        except Exception: pass
        try:
            dom_imgs = await self.page.locator("img, div[style*='background-image']").evaluate_all('''els => els.map(e => e.src || e.style.backgroundImage).filter(Boolean)''')
            for s in dom_imgs: self.interceptor.add_url(s, 'dom')
        except Exception: pass
        return self.interceptor.get_photos()

    async def scrape_cafe(self, cafe: CafeRecord):
        self.requests_count += 1
        if self.requests_count >= self.cfg.RECYCLE_CONTEXT_EVERY:
            await self._new_context()
        self.interceptor.clear()
        enc = quote_plus(cafe.name.strip())
        url = f'https://www.google.com/maps/search/{enc}/@{cafe.latitude},{cafe.longitude},{self.cfg.ZOOM_LEVEL}z?hl=fa'
        try:
            await self.page.goto(url, wait_until='domcontentloaded', timeout=30_000)
            sig = await self._wait_signal()
            if sig == 'captcha':
                print('⚠️ Cooldown triggered due to rate-limit signal...')
                await asyncio.sleep(60.0)
                await self._new_context()
                return 'rate_limited', []
            if sig == 'direct_place':
                h1_el = self.page.locator('h1.DUwDvf, h1').first
                title = (await h1_el.inner_text()).strip() if await h1_el.is_visible() else cafe.name
                m_coords = re.search(r'/@(-?\d+\.\d+),(-?\d+\.\d+)', self.page.url)
                plat = float(m_coords.group(1)) if m_coords else cafe.latitude
                plng = float(m_coords.group(2)) if m_coords else cafe.longitude
                matched, score, dist, nscore, _ = evaluate_candidate(cafe.name, cafe.latitude, cafe.longitude, title, plat, plng, self.cfg.MATCH_RADIUS_METERS, self.cfg.MATCH_THRESHOLD)
                if not matched:
                    self.db.save_place_and_photos(cafe, 'not_found', title, plat, plng, nscore, dist, url, self.page.url)
                    return 'not_found', []
                photos = await self._harvest_photos()
                self.db.save_place_and_photos(cafe, 'matched', title, plat, plng, nscore, dist, url, self.page.url, photos)
                return 'matched', photos
            elif sig == 'search_feed':
                links = self.page.locator("a[href*='/maps/place/']")
                count = await links.count()
                best, best_match = None, None
                for i in range(min(count, 10)):
                    link = links.nth(i)
                    cand_title = (await link.get_attribute('aria-label')) or ''
                    href = (await link.get_attribute('href')) or ''
                    m_c = re.search(r'!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)', href) or re.search(r'/@(-?\d+\.\d+),(-?\d+\.\d+)', href)
                    clat = float(m_c.group(1)) if m_c else None
                    clng = float(m_c.group(2)) if m_c else None
                    matched, score, dist, nscore, _ = evaluate_candidate(cafe.name, cafe.latitude, cafe.longitude, cand_title, clat, clng, self.cfg.MATCH_RADIUS_METERS, self.cfg.MATCH_THRESHOLD)
                    if matched and (not best_match or score > best_match[1]):
                        best_match = (matched, score, dist, nscore)
                        best = (link, cand_title, href, clat, clng)
                if not best:
                    self.db.save_place_and_photos(cafe, 'not_found', search_url=url)
                    return 'not_found', []
                await best[0].click()
                await self._wait_signal(5.0)
                photos = await self._harvest_photos()
                self.db.save_place_and_photos(cafe, 'matched', best[1], best[3], best[4], best_match[3], best_match[2], url, self.page.url, photos)
                return 'matched', photos
            else:
                self.db.save_place_and_photos(cafe, 'not_found', search_url=url)
                return 'not_found', []
        except Exception as e:
            self.db.save_place_and_photos(cafe, 'error', search_url=url)
            return 'error', []

print('✅ Scraper Orchestrator ready!')

In [ ]:
# 6. اجرای پایپ‌لاین استخراج تصاویر با لاگ و نوار پیشرفت (Execution Pipeline)
from tqdm.auto import tqdm

async def run_pipeline():
    db = DatabaseManager(config.SOURCE_DB_PATH, config.OUTPUT_DB_PATH)
    pending = db.load_pending_cafes(
        limit=config.LIMIT_CAFES,
        shard_id=config.SHARD_ID,
        total_shards=config.TOTAL_SHARDS
    )
    total = len(pending)
    print('=' * 60)
    print(f'🚀 MAPIFY KAGGLE HARVESTER STARTED')
    print(f'Source DB: {config.SOURCE_DB_PATH}')
    print(f'Output DB: {config.OUTPUT_DB_PATH}')
    if config.TOTAL_SHARDS and config.SHARD_ID:
        print(f'Shard: {config.SHARD_ID} of {config.TOTAL_SHARDS}')
    print(f'Pending Cafes to Scrape: {total}')
    print('=' * 60)
    
    if total == 0:
        print('✅ All cafes are already scraped!')
        return

    scraper = MapifyScraper(config, db)
    await scraper.start()

    matched, not_found, photos_total = 0, 0, 0
    pbar = tqdm(total=total, desc='Scraping Cafes')

    try:
        for idx, cafe in enumerate(pending, 1):
            status, photos = await scraper.scrape_cafe(cafe)
            if status == 'matched':
                matched += 1
                photos_total += len(photos)
            else:
                not_found += 1

            pbar.update(1)
            pbar.set_postfix({'matched': matched, 'not_found': not_found, 'photos': photos_total})

            # مکث بین درخواست‌ها و استراحت دوره‌ای
            if idx < total:
                if idx % config.CAFES_PER_BATCH == 0:
                    await asyncio.sleep(config.BATCH_COOLDOWN_SECONDS)
                else:
                    delay = random.uniform(config.MIN_DELAY_PER_CAFE, config.MAX_DELAY_PER_CAFE)
                    await asyncio.sleep(delay)

    finally:
        pbar.close()
        await scraper.close()

    print('=' * 60)
    print(f'📊 SESSION COMPLETE SUMMARY')
    print(f'Processed: {matched + not_found} / {total}')
    print(f'Matched: {matched}')
    print(f'Not Found: {not_found}')
    print(f'Total High-Res Photos Harvested: {photos_total}')
    print(f'Output SQLite Saved At: {config.OUTPUT_DB_PATH}')
    print('=' * 60)

# در محیط ژوپیتر/کگل، اجرای asyncio مستقیم با await یا create_task انجام می‌شود:
await run_pipeline()

In [ ]:
# 7. بررسی خروجی دیتابیس و نمایش نمونه‌ای از مکان‌ها همراه با شناسه‌های گوگل مپ
import sqlite3
import pandas as pd

conn = sqlite3.connect(config.OUTPUT_DB_PATH)

# نمایش ۵ مکان اول با ستون‌های کلیدی
df_places = pd.read_sql_query('''
SELECT cafe_id, title, place_id, cid, fid, kgmid, status, name_score, photos_count, scraped_at
FROM places
LIMIT 5;
''', conn)
print('📌 نمونه مکان‌های ثبت‌شده همراه با شناسه‌های یکتا:')
display(df_places)

# نمایش نمونه تصاویر باکیفیت استخراج‌شده
df_photos = pd.read_sql_query('''
SELECT cafe_id, source_type, high_res_url
FROM place_photos
LIMIT 5;
''', conn)
print('\n📸 نمونه لینک‌های باکیفیت (s0-rw):')
display(df_photos)

total_places = conn.execute('SELECT count(*) FROM places').fetchone()[0]
total_photos = conn.execute('SELECT count(*) FROM place_photos').fetchone()[0]
print(f'\n✅ مجموع کل مکان‌ها: {total_places} | مجموع کل عکس‌های باکیفیت: {total_photos}')
conn.close()